# Session 3 — zero-structure EDA + first scored baseline

1. Per-series zero table (mean_sales, zero_rate, weeks) — promoted from `01_practice.ipynb`.
2. Dead-series check (`zero_rate == 1.0`) — CONTEXT.md says this should be 0.
3. Lifecycle classification of each series' zeros: leading, trailing, interior, always-zero.
4. Score the MA12 baseline on the 13-week holdout with the official metric.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src.data import KEY
from src.backtest import split_last_horizon, ma12_forecast, HORIZON
from src.metrics import vn1_score

df = pd.read_parquet("../data/processed/vn1_long.parquet")
df.shape

(2559010, 7)

## 1. Per-series zero table

In [2]:
# Cell — per-series zero table
zero_table = df.groupby(KEY)["sales"].agg(
    mean_sales="mean",
    zero_rate=lambda s: (s == 0).mean(),
    weeks="count",
)
print("series:", len(zero_table))
zero_table.describe().round(3)

series: 15053


,mean_sales,zero_rate,weeks
count,15053.000,15053.000,15053.0
mean,12.684,0.718,170.0
std,82.230,0.297,0.0
min,0.006,0.000,170.0
25%,0.071,0.547,170.0
50%,0.559,0.835,170.0
75%,4.235,0.959,170.0
max,4808.841,0.994,170.0


In [3]:
# Cell — distribution of zero_rate across all series
# (bucketed counts, not a plot — matplotlib patch rendering is broken in this
# env right now; see PROGRESS.md note. Revisit once that's fixed.)
bins = [0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
labels = ["0-10%", "10-30%", "30-50%", "50-70%", "70-90%", "90-100%"]
bucketed = pd.cut(zero_table["zero_rate"], bins=bins, labels=labels, include_lowest=True)
print(bucketed.value_counts().sort_index())

print()
print("mean zero_rate  :", round(zero_table["zero_rate"].mean(), 3))
print("median zero_rate:", round(zero_table["zero_rate"].median(), 3))
print(">90% zero weeks :", (zero_table["zero_rate"] > 0.90).sum())
print("<10% zero weeks :", (zero_table["zero_rate"] < 0.10).sum())
print("max zero_rate   :", zero_table["zero_rate"].max())

zero_rate
0-10%      1021
10-30%     1007
30-50%     1398
50-70%     1919
70-90%     3694
90-100%    6014
Name: count, dtype: int64

mean zero_rate  : 0.718
median zero_rate: 0.835
>90% zero weeks : 6014
<10% zero weeks : 994
max zero_rate   : 0.9941176470588236


## 2. Dead-series check

In [4]:
# Cell — dead series: zero_rate == 1.0 (never sold at all)
dead = zero_table[zero_table["zero_rate"] == 1.0]
print("dead series (zero_rate == 1.0):", len(dead))
# Expect 0 — CONTEXT.md: every series has at least one positive sale.

dead series (zero_rate == 1.0): 0


## 3. Lifecycle classification

For each series, find the first and last week with a positive sale.
Zeros before the first sale are **leading** (pre-launch), zeros after the
last sale are **trailing** (discontinued), zeros between them are
**interior** (genuine intermittency).

In [5]:
# Cell — first/last positive-sale week per series
df_sorted = df.sort_values(KEY + ["date"])
positive = df_sorted[df_sorted["sales"] > 0]

first_last = positive.groupby(KEY)["date"].agg(first_sale="min", last_sale="max")
lifecycle = df_sorted.merge(first_last, on=KEY, how="left")

is_zero = lifecycle["sales"] == 0
leading = is_zero & (lifecycle["date"] < lifecycle["first_sale"])
trailing = is_zero & (lifecycle["date"] > lifecycle["last_sale"])
interior = is_zero & ~leading & ~trailing

counts = pd.Series({
    "leading (pre-launch)": leading.sum(),
    "trailing (discontinued)": trailing.sum(),
    "interior (intermittent)": interior.sum(),
    "always-zero (dead)": len(dead) * 170,
})
counts

leading (pre-launch)       1139262
trailing (discontinued)     230029
interior (intermittent)     467146
always-zero (dead)               0
dtype: int64

## 4. MA12 baseline, scored on the 13-week holdout

In [6]:
# Cell — hold out the last 13 weeks, forecast with MA12, score
train, test = split_last_horizon(df, horizon=HORIZON)
print("train dates:", train["date"].min(), "->", train["date"].max())
print("test dates :", test["date"].min(), "->", test["date"].max())
print("train weeks:", train["date"].nunique(), " test weeks:", test["date"].nunique())

train dates: 2020-07-06 00:00:00 -> 2023-07-03 00:00:00
test dates : 2023-07-10 00:00:00 -> 2023-10-02 00:00:00
train weeks: 157  test weeks: 13


In [7]:
# Cell — build forecast, align to test rows, score
forecast = ma12_forecast(train)
scored = test.merge(forecast, on=KEY, how="left")
assert scored["forecast"].isna().sum() == 0, "every series should have a forecast"

score = vn1_score(scored["sales"], scored["forecast"])
print(f"MA12 baseline VN1 score: {score:.4f}")

MA12 baseline VN1 score: 0.5163
